# 04 — Base de datos PostgreSQL: esquema relacional, carga y consultas

**Objetivo:** pasar de un archivo plano a una **base de datos relacional** con reglas de integridad, índices, vistas para Power BI y un usuario de solo lectura.

**Requisitos:** PostgreSQL instalado y el `.env` con `PG_PASSWORD`, `PG_BI_USER` y `PG_BI_PASSWORD`.

**Resultados que guarda:** base `cobranza` con 9 tablas y 2 vistas; `docs/validacion_carga.csv`.

## 1. Preparación

In [ ]:
import sys
from pathlib import Path

RAIZ = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd
from sqlalchemy import text

from src.config import RUTAS
from src.objetivo import asignar_conjunto
from src.bd import (crear_base_si_no_existe, obtener_motor, ejecutar_sql, crear_rol_lectura,
                    preparar_tablas, cargar_tablas, validar_carga)

## 2. Crear la base `cobranza`
Se conecta a la base de sistema `postgres` y crea `cobranza` solo si no existe.

In [ ]:
crear_base_si_no_existe()
motor = obtener_motor()
with motor.connect() as con:
    print(con.execute(text('SELECT version()')).scalar())

## 3. Crear el esquema (tablas, llaves y restricciones)
El script borra y recrea las tablas: así el notebook se puede ejecutar las veces que haga falta (**idempotente**).

In [ ]:
ejecutar_sql(motor, '01_esquema.sql')

## 4. Preparar los datos
Agregamos la etiqueta de **conjunto** (entrenamiento, validación, inmadura o excluida) y separamos la población en las 4 tablas de datos.

In [ ]:
pob = pd.read_parquet(RUTAS['data_interim'] / 'poblacion_objetivo.parquet')
pob['conjunto'] = asignar_conjunto(pob)
print(pob['conjunto'].value_counts().to_string())

tablas = preparar_tablas(pob)
for nombre, df in tablas.items():
    print(f'{nombre:25s} {df.shape[0]:>7,} filas x {df.shape[1]:>2} columnas')

## 5. Cargar a PostgreSQL
Todo en **una sola transacción**: si algo falla, no queda una carga a medias.

In [ ]:
cargar_tablas(motor, tablas)

## 6. Índices, vistas y permisos

In [ ]:
ejecutar_sql(motor, '02_indices.sql')
ejecutar_sql(motor, '03_vistas.sql')
crear_rol_lectura(motor)
ejecutar_sql(motor, '04_permisos.sql')

## 7. Validar la carga
Comparamos conteos y sumas de control entre pandas y PostgreSQL. Si no coinciden, algo se perdió en la carga.

📋 **Pega esta tabla.**

In [ ]:
validacion = validar_carga(motor, tablas)
validacion.to_csv(RUTAS['docs'] / 'validacion_carga.csv', index=False, encoding='utf-8-sig')
validacion

## 8. Probar que la integridad funciona
Intentamos insertar datos **inválidos** a propósito. La base debe **rechazarlos**: esa es la evidencia de que las reglas de calidad viven en la base y no solo en el código.

In [ ]:
pruebas = {
    'Crédito con deudor inexistente (llave foránea)':
        "INSERT INTO creditos (credito_id, loan_key, listing_key, deudor_id, fecha_originacion, plazo_meses, monto_original, estado_credito) VALUES (999999, 'X', 'X', 999999, '2010-01-01', 36, 1000, 'Chargedoff')",
    'Objetivo A distinto de 0 o 1 (CHECK)':
        'UPDATE resultados_recuperacion SET objetivo_a = 5 WHERE credito_id = 1',
    'Tasa de recuperación mayor a 100 % (CHECK)':
        'UPDATE resultados_recuperacion SET objetivo_b_tasa = 1.5 WHERE credito_id = 1',
    'Crédito duplicado (UNIQUE)':
        "INSERT INTO creditos SELECT * FROM creditos WHERE credito_id = 1",
}
for nombre, sql in pruebas.items():
    try:
        with motor.begin() as con:
            con.exec_driver_sql(sql)
        print(f'❌ {nombre}: la base lo aceptó')
    except Exception as e:
        print(f'✅ {nombre}: rechazado ({type(e.orig).__name__})')

## 9. Probar el usuario de solo lectura
Power BI y la aplicación se conectan con `lector_bi`. Debe poder leer las **vistas**, pero no las tablas con datos personales ni modificar nada.

In [ ]:
motor_bi = obtener_motor(rol='lector')
with motor_bi.connect() as con:
    n = con.execute(text('SELECT COUNT(*) FROM vw_cartera_cobranza')).scalar()
    print(f'✅ lector_bi lee vw_cartera_cobranza: {n:,} filas')

for sql in ['SELECT * FROM perfil_solicitud LIMIT 1', 'DELETE FROM segmentos']:
    try:
        with motor_bi.begin() as con:
            con.exec_driver_sql(sql)
        print(f'❌ lector_bi pudo ejecutar: {sql}')
    except Exception as e:
        print(f'✅ Bloqueado para lector_bi: {sql}')
motor_bi.dispose()

## 10. Consultas de negocio (`sql/05_consultas_ejemplo.sql`)
📋 **Pega las salidas de Q1, Q2, Q3 y Q5.**

In [ ]:
consultas = (RUTAS['sql'] / '05_consultas_ejemplo.sql').read_text(encoding='utf-8')
bloques = [b.strip() for b in consultas.split('-- Q')[1:]]

with motor.connect() as con:
    for bloque in bloques:
        titulo, sql = bloque.split('\n', 1)
        print(f'\n===== Q{titulo} =====')
        print(pd.read_sql(text(sql.strip().rstrip(';')), con).to_string(index=False))

## 11. Inventario final de la base

In [ ]:
with motor.connect() as con:
    objetos = pd.read_sql(text("""
        SELECT table_name AS objeto,
               CASE table_type WHEN 'VIEW' THEN 'vista' ELSE 'tabla' END AS tipo
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY tipo, objeto"""), con)
    objetos['filas'] = [con.execute(text(f'SELECT COUNT(*) FROM {o}')).scalar() for o in objetos['objeto']]
objetos